# Post-Fire Land Cover Trajectory and Hierarchical Classification (2023-2026)
### Digital Image Processing Pipeline: Biophysical Spectral Indices, Mathematical Morphology, Haralick GLCM Texture, and 2-Stage Hierarchical Classification

**Mata Kuliah:** Pengolahan Citra Digital (PCD) - D4 Teknik Informatika POLBAN  
**Kelompok 3B:**  
- Idotoho Reimon Simajuntak (241524047)  
- Ihsan Fauzi (241524048)  
- Muhammad Azzar MannalHadi (241524051)  

---
## Ringkasan Metodologi (Domain Model Terminology)
1. **Multi-Temporal Data Acquisition (get_data.py):** Citra Sentinel-2 L2A periode *Pre-Fire Baseline* (Mei-Juli 2023), *Active Fire Peak* El Nino (September-November 2023), *Post-Fire Recovery Series* (2024 Y1, 2025 Y2), dan *Current Evaluation State* (Juni-September 2026), disertai anomali termal *NASA FIRMS Brightness Temperature (T21)*, *JRC Global Surface Water Mask*, rujukan *Google Dynamic World V1*, dan LiDAR *NASA GEDI L2A (RH98)*.
2. **Method 1 (Burn Scar Delineation & Mathematical Morphology):** Perhitungan *Differenced Normalized Burn Ratio (dNBR / Burn Severity Index)* > 0.20, penyaringan *Pre-Fire NDVI* dan konfirmasi termal *FIRMS T21* > 310 K untuk membentuk *Raw Burn Candidate Mask*, dilanjutkan operasi *Morphological Opening* (5x5), *Morphological Closing* (7x7), dan *Connected Component Filtering* (< 1 Ha) guna menghasilkan *Clean Burn Scar Polygon*.
3. **Method 2 (Multi-Temporal Spectral & Haralick GLCM Feature Extraction):** Perhitungan indeks spektral biofisik (*NDVI*, *NDMI*, *BSI*) dan tekstur *Gray-Level Co-occurrence Matrix (GLCM)* Orde-2 Haralick (*Homogeneity*, *Contrast*, dan *Shannon Entropy*) pada 4 sudut arah (0, 45, 90, 135 derajat) sepanjang deret waktu 2023-2026.
4. **Method 3 (2-Stage Hierarchical Classification):**
   - **Stage 1 (Non-Vegetated Spectral Isolation):** Piksel tanah gambut terbuka dan arang pasca-bakar diisolasi secara spektral (*BSI > 0*, defisit *NDMI*) dan dipertahankan langsung sebagai **Class 4: Exposed Bare Peat / Unrecovered Land** (Merah) tanpa dipaksa masuk ke komputasi tekstur kanopi GLCM.
   - **Stage 2 (GLCM Canopy Clustering on Vegetated Pixels):** Khusus pada piksel bervegetasi, dijalankan *K-Means Clustering* (k=3) berbasis *[NDVI, NDMI, BSI, Homogeneity, Entropy]* untuk memisahkan struktur fisik kanopi:
     - **Class 1:** *Structured Monoculture Vegetation* (Kuning - Homogeneity tertinggi, Entropy terendah)
     - **Class 2:** *Degraded Shrubland* (Oranye - Tajuk semak terbuka, kelembaban terbatas)
     - **Class 3:** *Natural Forest Succession* (Hijau - Kanopi multi-strata heterogen, NDVI dan NDMI tertinggi)
5. **Independent Validation:** Evaluasi *4-Class Confusion Matrix* terhadap *Google Dynamic World V1 2026* (*Overall Accuracy* dan *Cohen's Kappa*) serta validasi struktur vertikal menggunakan *NASA GEDI L2A RH98 Canopy Height*.


## 1. Import Pustaka dan Konfigurasi
Seluruh fungsi pengolahan citra berada di `analisis_karhutla.py`. Wilayah utama menggunakan dataset master 30x30 km (`Kalteng_Sebangau_Kahayan`) beresolusi 10 meter.

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

from analisis_karhutla import load_dataset, run_pipeline, EPOCHS, RESULTS_DIR, DATA_DIR

ROI_NAME = "Kalteng_Sebangau_Kahayan"

## 2. Pemeriksaan Dataset Deret Waktu (2023 Pre - 2026 Y3)

In [ ]:
images, firms_t21, jrc_water, gt_dw, gedi_h, pixel_size_m = load_dataset(ROI_NAME)
for label, img in images.items():
    valid_pct = (img["b8"] > 0).mean() * 100
    print(f"{label:10s} | dimensi {img['b8'].shape} | resolusi {pixel_size_m:.0f}m | bebas awan {valid_pct:5.1f}%")
print("Footprint GEDI valid:", 0 if gedi_h is None else int((gedi_h > 0).sum()))

## 3. Eksekusi Pipeline Terpadu (Deliniasi, Morfologi, Spektral Makro, GLCM, K-Means, dan Evaluasi)

In [ ]:
df_stats, df_traj, oa, kappa = run_pipeline(ROI_NAME)
display(df_stats)

## 4. Figure 1: Sentinel-2 Multi-Spectral Composites and Biophysical Spectral Indices

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, "fig1_komposit_multispektral.png")))

## 5. Figure 2: Method 1 Pipeline (Burn Scar Delineation, FIRMS T21, Burn Severity dNBR, Morphological Opening & Closing)

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, "fig2_progres_workflow_morfologi.png")))

## 6. Figure 3: Method 2 Feature Extraction (NDVI, NDMI, BSI, GLCM Homogeneity, Contrast, Shannon Entropy)

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, "fig3_ekstraksi_fitur_spektral_glcm.png")))

## 7. Figure 4: Method 3 Comparison (Pre-GLCM Macro Spectral K-Means vs 2-Stage Hierarchical Spectral+GLCM)

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, "fig4_komparasi_sebelum_sesudah_glcm.png")))

## 8. Figure 5: 6-Panel Executive Summary Map (2023-2026 Chronological & 4-Class Hierarchical Map)

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, "hasil_analisis_pcd_karhutla.png")))

## 9. Figure 6: Multi-Temporal Trajectory Curves per Class (2023 Pre - 2026 Y3)

In [ ]:
display(df_traj.pivot_table(index="Epoch", columns="Nama_Kelas", values="Mean_NDVI", sort=False))
display(Image(filename=os.path.join(RESULTS_DIR, "trajektori_tahunan_2023_2026.png")))

## 10. Figure 7: 4-Class Confusion Matrix (Dynamic World V1 2026) and NASA GEDI L2A RH98 Canopy Height Validation

In [ ]:
print(f"Overall Accuracy: {oa:.2f}% | Cohen's Kappa: {kappa:.4f}")
display(Image(filename=os.path.join(RESULTS_DIR, "evaluasi_confusion_matrix_gedi.png")))